In [1]:
pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00


In [2]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [4]:
from clip_zeroshot import build_and_cache_text_features, build_and_cache_image_features, top_k_accuracy, load_cached_features

In [5]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


# Binning Probe on CLIP Zero Shot

### Prepare the dataset

In [6]:
from datasets import load_dataset

ds = load_dataset("axiong/imagenet-r")

README.md:   0%|          | 0.00/2.58k [00:00<?, ?B/s]

test/test-00000-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  446MB            

test/test-00000-of-00005.parquet: downloading bytes:           |  0.00B            

test/test-00001-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  424MB            

test/test-00001-of-00005.parquet: downloading bytes:           |  0.00B            

...

test/test-00003-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  459MB            

test/test-00003-of-00005.parquet: downloading bytes:           |  0.00B            

test/test-00004-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  417MB            

test/test-00004-of-00005.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/30000 [00:00<?, ? examples/s]

In [7]:
print(ds)
print(ds['test'].features)
print(ds["test"][0])
print(len(set(ds["test"]["class_name"])))

DatasetDict({
    test: Dataset({
        features: ['image', 'wnid', 'class_name'],
        num_rows: 30000
    })
})
{'image': Image(mode=None, decode=True), 'wnid': Value('string'), 'class_name': Value('string')}
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=570x622 at 0x7A98CF062660>, 'wnid': 'n02088094', 'class_name': 'afghan_hound'}
200


In [10]:
unique_pairs = sorted(set(zip(ds['test']["wnid"], ds['test']["class_name"])))
r_wnids = [pair[0] for pair in unique_pairs]
r_class_names = [pair[1].replace('_', ' ') for pair in unique_pairs]

print(len(r_wnids))
print(r_class_names[:5])

200
['goldfish', 'great white shark', 'hammerhead', 'stingray', 'hen']


In [13]:
wnid_to_r_index = {wnid: i for i, wnid in enumerate(r_wnids)}

### Prepare the Text Features

In [16]:
from imagenet_classes import IMAGENET_TEMPLATES

In [17]:
r_text_features = build_and_cache_text_features(model, tokenizer, r_class_names, IMAGENET_TEMPLATES, device, "./features", "imagenet_r_text_features")

  0%|          | 0/200 [00:00<?, ?it/s]

Text features has been saved at ./features/imagenet_r_text_features.pt


### Build the dataloader

In [18]:
class ImageNetRDataset(Dataset):
    def __init__(self, dataset, preprocess, wnid_to_index):
        self.dataset = dataset
        self.preprocess = preprocess
        self.wnid_to_index = wnid_to_index

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        image = preprocess((example["image"]))
        label = wnid_to_r_index[example["wnid"]]
        return image, label

In [19]:
r_dataset = ImageNetRDataset(ds['test'], preprocess, wnid_to_r_index)
r_dataloader = DataLoader(r_dataset, batch_size=32, num_workers=2)

In [20]:
images, labels = next(iter(r_dataloader))
print(images.shape, labels.shape)
print(labels[:5])

torch.Size([32, 3, 224, 224]) torch.Size([32])
tensor([40, 40, 40, 40, 40])


### Prepare the Image Features

In [24]:
r_image_cach = build_and_cache_image_features(model, device, r_dataloader, "./features", "imagenet-r")

  0%|          | 0/938 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/imagenet-r.pt


### Evaluation

In [28]:
r_image_features = r_image_cach['image_features'].to(device)
r_labels = r_image_cach['labels'].to(device)
r_text_features = r_text_features.to(device)

similarity = r_image_features @ r_text_features
logits = model.logit_scale.exp() * similarity
acc1, acc5 = top_k_accuracy(logits, r_labels, topk=(1, 5))

In [34]:
probs = torch.softmax(logits, dim=-1)
conf, pred = probs.max(dim=-1)

bins = torch.linspace(0, 1, 11)
for i in range(10):
    mask = (conf >= bins[i]) & (conf < bins[i+1])
    if mask.sum() == 0:
        continue
    acc = (pred[mask] == r_labels[mask]).float().mean().item()
    print(f"conf [{bins[i]:.1f},{bins[i+1]:.1f}): n={mask.sum().item():4d}  acc={acc:.3f}")

conf [0.0,0.1): n= 620  acc=0.106
conf [0.1,0.2): n=2072  acc=0.208
conf [0.2,0.3): n=2283  acc=0.328
conf [0.3,0.4): n=2225  acc=0.464
conf [0.4,0.5): n=2268  acc=0.545
conf [0.5,0.6): n=2229  acc=0.651
conf [0.6,0.7): n=2165  acc=0.763
conf [0.7,0.8): n=2385  acc=0.851
conf [0.8,0.9): n=3321  acc=0.922
conf [0.9,1.0): n=10432  acc=0.987


# Binning Probe on TPT

In [37]:
import torchvision.transforms as transforms

augment_transform = transforms.Compose([
    transforms.Lambda(lambda im: im.convert("RGB")),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)) # from clip
])

In [38]:
def generate_N_views(N, transform_fn, image):
  views = [transform_fn(image) for _ in range(N)]
  curr_img = preprocess(image)
  views.append(curr_img)
  views = torch.stack(views)

  return views

### Training Loop

In [39]:
from coop import PromptLearner, TextEncoderWrapper

In [40]:
prompt_learner = PromptLearner(model, device, 4, tokenizer, 512, r_class_names)
text_encoder = TextEncoderWrapper(model)

for param in model.parameters():
  param.requires_grad_(False)

print([n for n, p in prompt_learner.named_parameters() if p.requires_grad])

['ctx']


In [45]:
from tqdm.notebook import tqdm
import torch.nn.functional as F
from tpt import tpt_entropy_loss

correct = 0
total = 0
subset_size = 500
logit_scale = model.logit_scale.exp()

confidences = []
correctness = []

for i in tqdm(range(subset_size)):
    test_image = ds['test'][i]['image']
    true_label = wnid_to_r_index[ds['test'][i]['wnid']]

    prompt_learner.reset_context()
    optimizer = torch.optim.AdamW(prompt_learner.parameters(), lr=0.005)

    test_image_views = generate_N_views(63, augment_transform, test_image)
    image_features = model.encode_image(test_image_views.to(device))
    prompts, tok_prompts = prompt_learner()
    text_features = text_encoder(prompts, tok_prompts)
    text_features = text_features / text_features.norm(dim=-1,keepdim=True)

    logits = image_features @ text_features.t()
    loss = tpt_entropy_loss(logits)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
      clean_image = preprocess(test_image).unsqueeze(0).to(device)
      clean_feat = model.encode_image(clean_image)
      clean_feat = clean_feat / clean_feat.norm(dim=-1, keepdim=True)
      prompts, tok = prompt_learner()
      text_feat = text_encoder(prompts, tok)
      text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

      scaled_logits = logit_scale * (clean_feat @ text_feat.t())
      probs = torch.softmax(scaled_logits, dim=-1)
      conf, pred = probs.max(dim=-1)

    confidences.append(conf.item())
    correctness.append(pred.item() == true_label)
    correct += (pred.item() == true_label)
    total += 1

conf_t = torch.tensor(confidences)
correct_t = torch.tensor(correctness)

  0%|          | 0/500 [00:00<?, ?it/s]

In [46]:
accuracy = 100 * correct / total
accuracy

59.4

### Binning Probe

subset = 200;  
conf [0.00,0.50): n=  78  acc=0.333  
conf [0.50,0.70): n=  24  acc=0.708  
conf [0.70,0.85): n=  30  acc=0.800  
conf [0.85,1.00): n=  68  acc=0.971  

subset = 500;  
conf [0.00,0.50): n= 232  acc=0.289  
conf [0.50,0.70): n=  73  acc=0.740  
conf [0.70,0.85): n=  63  acc=0.810  
conf [0.85,1.00): n= 132  acc=0.947  

In [47]:
bins = torch.tensor([0.0, 0.5, 0.7, 0.85, 1.0])

for i in range(len(bins) - 1):
    mask = (conf_t >= bins[i]) & (conf_t < bins[i+1])
    n = mask.sum().item()
    if n == 0:
        continue
    acc = correct_t[mask].float().mean().item()
    print(f"conf [{bins[i]:.2f},{bins[i+1]:.2f}): n={n:4d}  acc={acc:.3f}")

conf [0.00,0.50): n= 232  acc=0.289
conf [0.50,0.70): n=  73  acc=0.740
conf [0.70,0.85): n=  63  acc=0.810
conf [0.85,1.00): n= 132  acc=0.947
